# 01 — Ground truth

**Qué hace.** Convierte los guiones y las tablas de `docs/guiones.md` en datos que el resto del pipeline puede leer, y reporta la distribución de cada campo.

**Qué lee.** `docs/guiones.md` (ruta en `config.yaml → rutas.guiones`) y la fecha de grabación de `config.yaml → proyecto.fecha_grabacion`.

**Qué produce.**
- `data/ground_truth/entrevistas.csv`: una fila por entrevista, con los seis campos, las trampas y la posición en que se dice cada campo.
- `data/ground_truth/llamadas.csv`: una fila por llamada. `items` va como JSON en una columna. Incluye las trampas plantadas para puntuar el validador.
- `data/referencias/e1.txt … l5.txt`: la transcripción de referencia. Es el texto del guion sin las etiquetas `A:`/`B:` ni formato markdown, con un turno por línea.
- `data/referencias/carta.csv` y `promociones.csv`: precios y descuentos del restaurante, para el validador aritmético.

**Por qué el ground truth cuesta cero.** El guion se escribió antes de grabar: la transcripción exacta y los valores correctos existen antes que el audio. No dependen de otro ASR (sin evaluación circular).

**Regla de mantenimiento.** Si al grabar se cambia una palabra, se corrige `docs/guiones.md` y se vuelve a ejecutar este notebook. Los CSV y los `.txt` nunca se editan a mano.

In [1]:
import json
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config

cfg = cargar_config()
rutas = cfg["rutas"]
FECHA_GRABACION = pd.Timestamp(cfg["proyecto"]["fecha_grabacion"])

texto = rutas["guiones"].read_text(encoding="utf-8")
print(f"{rutas['guiones'].relative_to(RAIZ)}: {len(texto.splitlines())} líneas")
print(f"Fecha de grabación: {FECHA_GRABACION.date()} ({FECHA_GRABACION.day_name()})")

docs/guiones.md: 604 líneas
Fecha de grabación: 2026-10-31 (Saturday)


## 1. Leer el documento

Funciones mínimas para partir el markdown en secciones y tablas. Solo viven aquí porque solo este notebook lee `guiones.md`; el resto del pipeline lee los CSV.

In [2]:
def secciones(texto, nivel):
    """{título: cuerpo} para los encabezados de un nivel (2 = '##', 3 = '###')."""
    marca = "#" * nivel + " "
    partes = re.split(rf"^{marca}(.+)$", texto, flags=re.M)
    return {partes[i].strip(): partes[i + 1] for i in range(1, len(partes), 2)}


def tablas(cuerpo):
    """Todas las tablas markdown de un bloque, como DataFrames de texto."""
    resultado, actual = [], []
    for linea in cuerpo.splitlines() + [""]:
        if linea.strip().startswith("|"):
            actual.append([c.strip() for c in linea.strip().strip("|").split("|")])
        elif actual:
            encabezado, _separador, *filas = actual
            resultado.append(pd.DataFrame(filas, columns=encabezado))
            actual = []
    return resultado


def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


sec = secciones(texto, 2)
list(sec)

['Cómo usar estos guiones',
 'Task 1 — Entrevistas',
 'Task 1 — Ground truth',
 'Task 2 — Carta del restaurante',
 'Task 2 — Llamadas',
 'Task 2 — Ground truth y trampas']

## 2. Transcripciones de referencia

Cada turno del guion tiene la forma `**A:** texto`. Se quita la etiqueta del hablante y cualquier marca de markdown. La puntuación y las mayúsculas se dejan tal cual: la normalización para el WER se aplica después, igual a referencia e hipótesis (`src/normalizacion.py`). Así el `.txt` sigue siendo legible.

In [3]:
def turnos(cuerpo):
    """Texto de cada turno del diálogo, sin 'A:'/'B:' ni markdown."""
    salida = []
    for linea in cuerpo.splitlines():
        m = re.match(r"^\*\*([AB]):\*\*\s*(.+)$", linea.strip())
        if m:
            salida.append(re.sub(r"[*`]", "", m.group(2)).strip())
    return salida


guiones = {}
for nombre_seccion in ["Task 1 — Entrevistas", "Task 2 — Llamadas"]:
    for titulo, cuerpo in secciones(sec[nombre_seccion], 3).items():
        clip, descripcion = (s.strip() for s in titulo.split("—", 1))
        guiones[clip.lower()] = {"titulo": descripcion, "turnos": turnos(cuerpo)}

rutas["referencias_dir"].mkdir(parents=True, exist_ok=True)
ppm = cfg["proyecto"]["palabras_por_minuto_estimadas"]
dur_min, dur_max = cfg["proyecto"]["duracion_clip_min"]
filas = []
for clip, g in guiones.items():
    referencia = "\n".join(g["turnos"]) + "\n"
    assert not re.search(r"\b[AB]:|\*", referencia), f"{clip}: quedan etiquetas o markdown"
    destino = rutas["referencias_dir"] / cfg["archivos"]["patron_referencia"].format(clip=clip)
    destino.write_text(referencia, encoding="utf-8")
    palabras = len(referencia.split())
    filas.append({"clip": clip, "titulo": g["titulo"], "turnos": len(g["turnos"]),
                  "palabras": palabras, "min_estimados": round(palabras / ppm, 1),
                  "archivo": str(destino.relative_to(RAIZ))})

referencias = pd.DataFrame(filas)
if rutas["duraciones"].exists():
    medidas = pd.read_csv(rutas["duraciones"])[["clip", "duracion_min"]]
    referencias = referencias.merge(medidas, on="clip", how="left")
    referencias["en_rango"] = referencias.duracion_min.between(dur_min, dur_max)
    print("en_rango usa la duración medida sobre cada audio en 02_transcripcion_gpu")
else:
    referencias["en_rango"] = referencias.min_estimados.between(dur_min, dur_max)
    print(f"Todavía no existe {rutas['duraciones'].relative_to(RAIZ)}: en_rango usa la estimación por palabras")
referencias

en_rango usa la duración medida sobre cada audio en 02_transcripcion_gpu


,clip,titulo,turnos,palabras,min_estimados,archivo,duracion_min,en_rango
0,e1,Desarrollador backend,22,377,2.5,data/referencias/e1.txt,2.38,True
1,e2,Analista de datos,20,337,2.2,data/referencias/e2.txt,2.11,True
2,e3,Jefe de infraestructura,20,301,2.0,data/referencias/e3.txt,1.97,False
3,e4,Desarrollador frontend,19,260,1.7,data/referencias/e4.txt,2.14,True
4,e5,Líder de QA,18,285,1.9,data/referencias/e5.txt,2.23,True
5,l1,Pedido familiar con Dúo,25,275,1.8,data/referencias/l1.txt,2.18,True
6,l2,Dos medios que se vuelven un entero,21,228,1.5,data/referencias/l2.txt,2.01,True
7,l3,Tres combos y una dirección corregida,21,208,1.4,data/referencias/l3.txt,2.11,True
8,l4,Almuerzo de oficina,20,207,1.4,data/referencias/l4.txt,2.36,True
9,l5,Reunión familiar,21,207,1.4,data/referencias/l5.txt,2.17,True


In [4]:
print((rutas["referencias_dir"] / "e1.txt").read_text(encoding="utf-8")[:400])

Buenas tardes, gracias por venir. Soy Carlos, del área de talento de Andina Software. Para empezar, cuéntame un poco de ti y de lo que haces actualmente.
Buenas tardes, gracias por la invitación. Bueno, actualmente trabajo en una fintech pequeña que se llama Crédito Ágil. Ahí estoy como desarrollador backend semi senior. Mi trabajo es básicamente construir y mantener las APIs que usan la app móvil


`min_estimados` usa las palabras por minuto de `config.yaml → proyecto.palabras_por_minuto_estimadas`: fue solo una guía antes de grabar. La duración real la mide `02_transcripcion_gpu` sobre cada audio y la guarda en `data/resultados/duraciones.csv`. Cuando ese archivo existe, la tabla la trae en `duracion_min` y `en_rango` se calcula con ella.

## 3. Ground truth de entrevistas (Task 1)

Se separan las celdas compuestas de la tabla:
- `6500 PEN` → `pretension_salarial = 6500` y `moneda = PEN`.
- `2026-11-15 ("en dos semanas")` → `disponibilidad = 2026-11-15` y `disponibilidad_frase = en dos semanas`. La frase es lo que de verdad se dice en el audio.

`tecnologias` se guarda como lista JSON. Las posiciones (1 = el campo se dice primero) quedan como columnas `pos_*`, para mostrar cómo varía el orden.

In [5]:
gt1, trampas1, posiciones = tablas(sec["Task 1 — Ground truth"])


def separar_fecha(celda):
    m = re.match(r"(\d{4}-\d{2}-\d{2})\s*\((.*)\)\s*$", celda)
    return m.group(1), m.group(2).replace('"', "").strip()


fechas = gt1.disponibilidad.map(separar_fecha)
salario = gt1.pretension_salarial.str.split()

entrevistas = pd.DataFrame({
    "clip": gt1.Clip.str.lower(),
    "anios_experiencia": gt1.anios_experiencia.astype(float),
    "ultimo_puesto": gt1.ultimo_puesto,
    "tecnologias": gt1.tecnologias.map(
        lambda s: json.dumps([t.strip() for t in s.split(",")], ensure_ascii=False)),
    "pretension_salarial": salario.str[0].astype(int),
    "moneda": salario.str[1],
    "disponibilidad": fechas.str[0],
    "disponibilidad_frase": fechas.str[1],
    "nivel_ingles": gt1.nivel_ingles,
    "trampas": trampas1.set_index("Clip").Trampas.reindex(gt1.Clip).values,
})
pos = posiciones.set_index("Campo").T.astype(int).add_prefix("pos_")
pos.index = pos.index.str.lower()
entrevistas = entrevistas.merge(pos, left_on="clip", right_index=True)

esquema = cfg["esquemas"]["entrevista"]
assert set(entrevistas.nivel_ingles) <= set(esquema["clases"]["nivel_ingles"])
assert set(entrevistas.moneda) <= set(esquema["monedas"])
assert (pd.to_datetime(entrevistas.disponibilidad) >= FECHA_GRABACION).all()
assert set(entrevistas["clip"]) == {c for c in guiones if c.startswith("e")}

rutas["ground_truth_entrevistas"].parent.mkdir(parents=True, exist_ok=True)
entrevistas.to_csv(rutas["ground_truth_entrevistas"], index=False)
entrevistas

,clip,anios_experiencia,ultimo_puesto,tecnologias,pretension_salarial,moneda,disponibilidad,disponibilidad_frase,nivel_ingles,trampas,pos_ultimo_puesto,pos_anios_experiencia,pos_tecnologias,pos_nivel_ingles,pos_pretension_salarial,pos_disponibilidad
0,e1,4.5,Desarrollador backend semi senior,"[""Python"", ""Django"", ""PostgreSQL"", ""Docker""]",6500,PEN,2026-11-14,en dos semanas,intermedio,"""casi cinco"" antes de ""cuatro años y medio""; s...",1,2,3,4,5,6
1,e2,3.0,Analista de datos senior,"[""Power BI"", ""SQL Server"", ""Python"", ""pandas"",...",1800,USD,2026-10-31,de inmediato,avanzado,"años dichos como ""desde dos mil veintitrés""; m...",4,3,5,2,6,1
2,e3,8.0,Jefe de infraestructura,"[""AWS"", ""Kubernetes"", ""Terraform"", ""Jenkins"", ...",9500,PEN,2026-12-01,primero de diciembre,basico,"rango ""entre nueve y diez mil"" con ideal; ""cas...",2,3,4,5,1,6
3,e4,2.5,Desarrollador frontend junior,"[""React"", ""TypeScript"", ""Next.js"", ""Figma"", ""J...",4800,PEN,2026-11-30,un mes de preaviso,intermedio,"los 2.5 años nunca se dicen, hay que sumar: de...",3,2,1,5,4,6
4,e5,6.0,Líder técnico de QA,"[""Cypress"", ""Selenium"", ""Postman"", ""Jira"", ""JM...",7000,PEN,2027-01-11,el lunes once de enero,avanzado,sueldo actual (6200) antes de la pretensión; a...,2,6,3,1,5,4


## 4. Carta y promociones (Task 2)

El validador aritmético necesita los precios. Se extraen de la misma fuente, `guiones.md`, para que solo exista una versión de la carta.

In [6]:
carta = tablas(sec["Task 2 — Carta del restaurante"])[0]
carta.columns = ["producto", "tamano", "precio"]
carta = carta.assign(producto=carta.producto.str.lower(),
                     tamano=carta.tamano.str.lower(),
                     precio=carta.precio.astype(int))

promociones = pd.DataFrame(
    [{"promocion": sin_tildes(n).lower(), "descuento": int(d), "condicion": c.strip(" .")}
     for n, d, c in re.findall(r"^- \*\*(.+?):\*\*\s*S/\s*(\d+) de descuento (.+)$",
                               sec["Task 2 — Carta del restaurante"], flags=re.M)])

carta.to_csv(rutas["carta"], index=False)
promociones.to_csv(rutas["promociones"], index=False)
display(carta)
promociones

,producto,tamano,precio
0,pollo a la brasa,entero,68
1,pollo a la brasa,medio,36
2,pollo a la brasa,cuarto,20
3,papas fritas,familiar,14
4,papas fritas,personal,7
5,chicha morada,jarra de 1 litro,12
6,chicha morada,vaso,5
7,gaseosa,1.5 litros,9
8,gaseosa,personal,4
9,ensalada clásica,único,10


,promocion,descuento,condicion
0,duo familiar,10,si el pedido incluye un pollo entero y unas pa...
1,promo app,5,si el cliente da el código LEÑA5


## 5. Ground truth de llamadas (Task 2)

Cada ítem de la tabla (`Pollo a la brasa entero 1 × 68`) se separa en producto, tamaño, cantidad y precio, y **se contrasta con la carta**. Si un precio del ground truth no coincide con la carta, el notebook falla. También se comprueba que `total correcto = Σ cantidad × precio − descuento`, y que la diferencia con `total_dicho` es la que declara la tabla de trampas.

In [7]:
gt2, trampas2 = tablas(sec["Task 2 — Ground truth y trampas"])
PRODUCTOS = sorted(carta.producto.unique(), key=len, reverse=True)


def parsear_items(celda):
    items = []
    for parte in celda.split(";"):
        descripcion, cantidad, precio = re.match(r"^(.+?)\s+(\d+)\s*×\s*(\d+)$", parte.strip()).groups()
        producto = next(p for p in PRODUCTOS if descripcion.lower().startswith(p))
        opciones = carta[carta.producto == producto]
        tamano = descripcion[len(producto):].strip().lower() or opciones.tamano.iloc[0]
        fila = opciones[opciones.tamano.str.startswith(tamano)]
        assert len(fila) == 1 and fila.precio.iloc[0] == int(precio), f"no cuadra con la carta: {parte}"
        items.append({"producto": producto, "tamano": tamano,
                      "cantidad": int(cantidad), "precio_unitario": int(precio)})
    return items


def parsear_promocion(celda):
    if celda == "null":
        return None, 0
    nombre, descuento = re.match(r"^(.+?)\s*\(\s*[−-](\d+)\)$", celda).groups()
    return sin_tildes(nombre).lower(), int(descuento)


def vacio(celda):
    return None if celda.strip() in {"—", "-", ""} else celda.strip()


def diferencia(celda):
    m = re.search(r"\(([+−-])(\d+)\)", celda)
    return 0 if not m else int(m.group(2)) * (-1 if m.group(1) in "−-" else 1)


items = gt2.iloc[:, 1].map(parsear_items)
promo = gt2.promocion.map(parsear_promocion)
t2 = trampas2.set_index("Clip").reindex(gt2.Clip)

llamadas = pd.DataFrame({
    "clip": gt2.Clip.str.lower(),
    "items": items.map(lambda x: json.dumps(x, ensure_ascii=False)),
    "n_items": items.map(len),
    "promocion": promo.str[0],
    "descuento": promo.str[1],
    "direccion_entrega": gt2.direccion_entrega,
    "metodo_pago": gt2.metodo_pago,
    "total_dicho": gt2.total_dicho.astype(int),
    "total_correcto": gt2["Total correcto"].astype(int),
    "trampa_correccion": t2["Corrección del cliente"].map(vacio).notna().values,
    "correccion_cliente": t2["Corrección del cliente"].map(vacio).values,
    "trampa_total_errado": t2["Total errado"].str.startswith("Sí").values,
    "diferencia_total": t2["Total errado"].map(diferencia).values,
    "causa_error": t2["Causa real del error"].map(vacio).values,
    "trampa_direccion_digitos": gt2.direccion_entrega.str.contains(r"\d"),
    "distractores": t2["Distractores numéricos"].values,
})

suma = items.map(lambda xs: sum(i["cantidad"] * i["precio_unitario"] for i in xs))
assert (suma - llamadas.descuento == llamadas.total_correcto).all(), "total correcto mal calculado"
assert (llamadas.total_dicho - llamadas.total_correcto == llamadas.diferencia_total).all()
assert (llamadas.trampa_total_errado == (llamadas.diferencia_total != 0)).all()
assert set(llamadas.metodo_pago) <= set(cfg["esquemas"]["pedido"]["clases"]["metodo_pago"])
assert set(llamadas.promocion.dropna()) <= set(promociones.promocion)
assert set(llamadas["clip"]) == {c for c in guiones if c.startswith("l")}

rutas["ground_truth_llamadas"].parent.mkdir(parents=True, exist_ok=True)
llamadas.to_csv(rutas["ground_truth_llamadas"], index=False)
llamadas

,clip,items,n_items,promocion,descuento,direccion_entrega,metodo_pago,total_dicho,total_correcto,trampa_correccion,correccion_cliente,trampa_total_errado,diferencia_total,causa_error,trampa_direccion_digitos,distractores
0,l1,"[{""producto"": ""pollo a la brasa"", ""tamano"": ""e...",3,duo familiar,10,"Calle Los Cedros 245, departamento 302, Surco",yape,96,96,True,1 jarra → 2 jarras,False,0,NaN,True,"teléfono de 9 dígitos, ""cuarenta y cincuenta m..."
1,l2,"[{""producto"": ""pollo a la brasa"", ""tamano"": ""e...",3,promo app,5,"Avenida Las Gardenias 1580, Pueblo Libre",efectivo,95,90,True,2 medios pollos → 1 entero,True,5,Hablante: el vendedor olvidó el descuento,True,"""setenta y dos"", billete de cien"
2,l3,"[{""producto"": ""pollo a la brasa"", ""tamano"": ""c...",4,None,0,"Jirón Los Halcones 1034, interior 5, Jesús María",tarjeta,121,111,True,Dirección 1043 → 1034,True,10,Hablante: el vendedor sumó mal,True,número viejo de la dirección
3,l4,"[{""producto"": ""pollo a la brasa"", ""tamano"": ""m...",5,promo app,5,"Calle Las Orquídeas 418, oficina 702, Miraflores",plin,85,85,True,"2 mollejitas → 1, y agrega anticuchos",False,0,NaN,True,"""treinta y cinco minutos"", oficina 702"
4,l5,"[{""producto"": ""pollo a la brasa"", ""tamano"": ""e...",4,duo familiar,10,"Avenida Los Laureles 2271, bloque C, departame...",yape,131,140,True,2 enteros → 1 entero + 1 medio,True,-9,Hablante: el vendedor omitió una gaseosa,True,números de la dirección


In [8]:
print(json.dumps(json.loads(llamadas["items"][0]), ensure_ascii=False, indent=2))

[
  {
    "producto": "pollo a la brasa",
    "tamano": "entero",
    "cantidad": 1,
    "precio_unitario": 68
  },
  {
    "producto": "papas fritas",
    "tamano": "familiar",
    "cantidad": 1,
    "precio_unitario": 14
  },
  {
    "producto": "chicha morada",
    "tamano": "jarra",
    "cantidad": 2,
    "precio_unitario": 12
  }
]


### Requisitos del enunciado sobre las trampas

In [9]:
pd.DataFrame([
    ("Llamadas con autocorrección del cliente", 3, llamadas.trampa_correccion.sum()),
    ("Llamadas con total dicho errado", 3, llamadas.trampa_total_errado.sum()),
    ("Direcciones o contratos con dígitos", 1, llamadas.trampa_direccion_digitos.sum()),
], columns=["requisito", "mínimo", "corpus"]).assign(cumple=lambda d: d.corpus >= d.mínimo)

,requisito,mínimo,corpus,cumple
0,Llamadas con autocorrección del cliente,3,5,True
1,Llamadas con total dicho errado,3,3,True
2,Direcciones o contratos con dígitos,1,5,True


## 6. Distribución de cada campo

### Entrevistas

In [10]:
entrevistas[["anios_experiencia"]].describe().T

,count,mean,std,min,25%,50%,75%,max
anios_experiencia,5.0,4.8,2.252776,2.5,3.0,4.5,6.0,8.0


In [11]:
entrevistas.groupby("moneda").pretension_salarial.agg(["count", "min", "median", "max"])

,count,min,median,max
moneda,,,,
PEN,4,4800,6750.0,9500
USD,1,1800,1800.0,1800


In [12]:
clases_ingles = cfg["esquemas"]["entrevista"]["clases"]["nivel_ingles"]
entrevistas.nivel_ingles.value_counts().reindex(clases_ingles, fill_value=0).rename("clips").to_frame()

,clips
nivel_ingles,
basico,1
intermedio,2
avanzado,2


In [13]:
tecs = entrevistas.tecnologias.map(json.loads)
print(f"Tecnologías por clip: {tecs.map(len).tolist()}  ·  distintas en total: {tecs.explode().nunique()}")
tecs.explode().value_counts().rename("clips").to_frame().T

Tecnologías por clip: [4, 5, 5, 5, 5]  ·  distintas en total: 23


tecnologias,Python,Django,PostgreSQL,Docker,Power BI,SQL Server,pandas,Airflow,AWS,Kubernetes,...,React,TypeScript,Next.js,Figma,Jest,Cypress,Selenium,Postman,Jira,JMeter
clips,2,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [14]:
dias = (pd.to_datetime(entrevistas.disponibilidad) - FECHA_GRABACION).dt.days
entrevistas[["clip", "disponibilidad_frase", "disponibilidad"]].assign(dias_desde_grabacion=dias)

,clip,disponibilidad_frase,disponibilidad,dias_desde_grabacion
0,e1,en dos semanas,2026-11-14,14
1,e2,de inmediato,2026-10-31,0
2,e3,primero de diciembre,2026-12-01,31
3,e4,un mes de preaviso,2026-11-30,30
4,e5,el lunes once de enero,2027-01-11,72


In [15]:
entrevistas[["clip", "ultimo_puesto"]].assign(palabras=entrevistas.ultimo_puesto.str.split().map(len))

,clip,ultimo_puesto,palabras
0,e1,Desarrollador backend semi senior,4
1,e2,Analista de datos senior,4
2,e3,Jefe de infraestructura,3
3,e4,Desarrollador frontend junior,3
4,e5,Líder técnico de QA,4


**Cómo varía la posición de cada campo** (1 = se dice primero). Ningún campo ocupa siempre el mismo lugar, así que un extractor que aprenda "el salario va al final" no funciona.

In [16]:
tabla_pos = entrevistas.set_index("clip").filter(like="pos_").T
tabla_pos.index = tabla_pos.index.str.removeprefix("pos_")
tabla_pos.assign(posiciones_distintas=tabla_pos.nunique(axis=1))

clip,e1,e2,e3,e4,e5,posiciones_distintas
ultimo_puesto,1,4,2,3,2,4
anios_experiencia,2,3,3,2,6,3
tecnologias,3,5,4,1,3,4
nivel_ingles,4,2,5,5,1,4
pretension_salarial,5,6,1,4,5,4
disponibilidad,6,1,6,6,4,3


### Llamadas

In [17]:
llamadas.metodo_pago.value_counts().rename("llamadas").to_frame().T

metodo_pago,yape,efectivo,tarjeta,plin
llamadas,2,1,1,1


In [18]:
llamadas.promocion.fillna("(ninguna)").value_counts().rename("llamadas").to_frame().T

promocion,duo familiar,promo app,(ninguna)
llamadas,2,2,1


In [19]:
llamadas[["n_items", "total_dicho", "total_correcto"]].describe().T[["min", "50%", "max"]]

,min,50%,max
n_items,3.0,4.0,5.0
total_dicho,85.0,96.0,131.0
total_correcto,85.0,96.0,140.0


In [20]:
productos = llamadas["items"].map(json.loads).explode().map(lambda i: f"{i['producto']} ({i['tamano']})")
productos.value_counts().rename("llamadas").to_frame()

,llamadas
items,
pollo a la brasa (entero),3
papas fritas (familiar),2
anticuchos (porción),2
gaseosa (1.5 litros),2
pollo a la brasa (medio),2
chicha morada (jarra),1
pollo a la brasa (cuarto),1
papas fritas (personal),1
gaseosa (personal),1


In [21]:
llamadas.set_index("clip")[["trampa_correccion", "trampa_total_errado", "diferencia_total",
                              "causa_error", "trampa_direccion_digitos"]]

,trampa_correccion,trampa_total_errado,diferencia_total,causa_error,trampa_direccion_digitos
clip,,,,,
l1,True,False,0,NaN,True
l2,True,True,5,Hablante: el vendedor olvidó el descuento,True
l3,True,True,10,Hablante: el vendedor sumó mal,True
l4,True,False,0,NaN,True
l5,True,True,-9,Hablante: el vendedor omitió una gaseosa,True


### Comprobación contra lo declarado en `guiones.md`

El documento dice: *"nivel_ingles tiene 1 básico, 2 intermedios y 2 avanzados; la moneda es PEN en 4 clips y USD en 1; los años van de 2.5 a 8"*.

In [22]:
assert entrevistas.nivel_ingles.value_counts().to_dict() == {"intermedio": 2, "avanzado": 2, "basico": 1}
assert entrevistas.moneda.value_counts().to_dict() == {"PEN": 4, "USD": 1}
assert (entrevistas.anios_experiencia.min(), entrevistas.anios_experiencia.max()) == (2.5, 8.0)
print("La distribución coincide con lo declarado en guiones.md")

La distribución coincide con lo declarado en guiones.md


## Qué dice esta distribución sobre lo que el corpus puede medir

- **nivel_ingles**: 1 / 2 / 2 entre tres clases. Con un solo clip `basico`, un error en E3 hace caer el F1 de esa clase de 1 a 0 y arrastra mucho el macro F1. Hay que reportar ese número con su intervalo y decir que no es estable.
- **pretension_salarial**: un solo caso en USD. El corpus no permite afirmar nada sobre cómo se comporta la moneda.
- **anios_experiencia**: dos valores no enteros (4.5 y 2.5). En E4 el valor nunca se dice (hay que sumar 1.5 + 1), así que la regla de verificación debe dejarlo `sin_verificar` aunque sea correcto.
- **Llamadas**: las cinco tienen una corrección del cliente y tres tienen un total errado. Los tres errores son del hablante. Los errores del ASR aparecerán al degradar el audio (`04_degradacion`), y solo entonces habrá casos de las dos causas para probar la regla de diagnóstico.
- **Duración**: por número de palabras, varios guiones quedaban por debajo de 2 minutos, y las llamadas eran las más cortas. Esa estimación no basta. La duración que cuenta es la medida sobre cada audio en `02_transcripcion_gpu`: es la que usa `en_rango` en la sección 2 cuando existe `data/resultados/duraciones.csv`.
- 5 + 5 clips: alcanza para ver un efecto, no para dar un número preciso.